In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd
import re
from bs4 import BeautifulSoup

In [11]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# URL de base
base_url = "https://www.optioncarriere.tn/emploi?s=&l=Tunisie"

# Variables pour stocker les données
titles = []
companies = []
locations = []
dates = []
descriptions = []
links = []

# Fonction pour extraire les informations d'une seule page
def scrape_page(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Trouver toutes les offres d'emploi sur la page
    job_cards = soup.find_all('article', class_='job clicky')  # Classe des offres d'emploi
    
    for job in job_cards:
        # Titre de l'offre
        title = job.find('h2', class_='job-title')
        titles.append(title.get_text(strip=True) if title else None)
        
        # Nom de l'entreprise
        company = job.find('p', class_='company')
        companies.append(company.get_text(strip=True) if company else None)
        
        # Localisation
        location = job.find('ul', class_='location')
        locations.append(location.get_text(strip=True) if location else None)
        
        # Date de publication
        date = job.find('span', class_='badge badge-r badge-s')
        dates.append(date.get_text(strip=True) if date else None)
        
        # Lien vers l'offre
        link = job_card.find('a')  # Trouver la balise <a>
        if link:
            link_text = link.get_text(strip=True)  # Texte visible du lien
            href = link['href']  # URL partielle
            full_url = f"https://www.optioncarriere.tn{href}"  # Construire l'URL complète
            title = link.get('title', 'N/A')  # Attribut title
        else:
            link_text = None
            full_url = None
            title = None
        
        # Description (récupérer depuis la page individuelle de l'offre)
        if  full_url:
            job_response = requests.get( full_url)
            job_soup = BeautifulSoup(job_response.text, 'html.parser')
            description = job_soup.find('div', class_='job-description')
            descriptions.append(description.get_text(strip=True) if description else None)
        else:
            descriptions.append(None)

# Scraper toutes les pages
current_url = base_url
while current_url:
    print(f"Scraping: {current_url}")
    scrape_page(current_url)
    
    # Trouver le lien "Page suivante"
    response = requests.get(current_url)
    soup = BeautifulSoup(response.text, 'html.parser')
    next_page = soup.find('a', class_='next')
    if next_page and 'href' in next_page.attrs:
        current_url = f"https://www.optioncarriere.tn{next_page['href']}"
        time.sleep(2)  # Pause pour éviter d'être bloqué
    else:
        break  # Pas de page suivante

# Créer un DataFrame
jobs_df = pd.DataFrame({
    'Title': titles,
    'Company': companies,
    'Location': locations,
    'Date': dates,
    'Description': descriptions,
    'Link': links
})

# Sauvegarder les données dans un fichier CSV
jobs_df.to_csv('optioncarriere_jobs.csv', index=False)

print("Données sauvegardées dans 'optioncarriere_jobs.csv'")


Scraping: https://www.optioncarriere.tn/emploi?s=&l=Tunisie
Données sauvegardées dans 'optioncarriere_jobs.csv'


In [7]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

# Initialiser le driver Selenium
driver = webdriver.Chrome()  # Assurez-vous que le chromedriver est dans votre PATH
driver.get("https://www.optioncarriere.tn/emploi?s=&l=Tunisie")

# Variables pour stocker les données
titles = []
companies = []
locations = []
dates = []
links = []
descriptions = []

# Fonction pour extraire les informations d'une page
def scrape_page():
    try:
        # Trouver toutes les cartes d'offres d'emploi
        job_cards = driver.find_elements(By.CLASS_NAME, "job")  # Remplacez "job" par la vraie classe de chaque carte

        for card in job_cards:
            # Titre de l'offre
            try:
                title = card.find_element(By.XPATH, ".//a[@title]").get_attribute("title")
            except:
                title = "N/A"
            titles.append(title)

            # Nom de la compagnie
            try:
                company = card.find_element(By.CLASS_NAME, "company").text  # Remplacez "company" par la vraie classe
            except:
                company = "N/A"
            companies.append(company)

            # Localisation
            try:
                location = card.find_element(By.CLASS_NAME, "location").text  # Remplacez "location" par la vraie classe
            except:
                location = "N/A"
            locations.append(location)

            # Date de publication
            try:
                date = WebDriverWait(driver, 10).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, ".badge.badge-r.badge-s"))  # Modifier le sélecteur si nécessaire
                )
                offer_date = date.text
            except:
                date = "N/A"
            dates.append(offer_date)

            # Lien de l'offre
            try:
                link = card.find_element(By.XPATH, ".//a[@title]").get_attribute("href")
            except:
                link = "N/A"
            links.append(link)

            # Description de l'offre
            if link != "N/A":
                driver.execute_script("window.open('');")  # Ouvrir un nouvel onglet
                driver.switch_to.window(driver.window_handles[1])  # Passer au nouvel onglet
                driver.get(link)
                try:
                    description = WebDriverWait(driver, 10).until(
                        EC.presence_of_element_located((By.CLASS_NAME, "content"))  # Remplacez "job-description" par la vraie classe
                    ).text
                except:
                    description = "N/A"
                driver.close()  # Fermer l'onglet
                driver.switch_to.window(driver.window_handles[0])  # Retourner à l'onglet principal
            else:
                description = "N/A"
            descriptions.append(description)

    except Exception as e:
        print(f"Erreur lors du scraping de la page : {e}")

# Boucle pour parcourir toutes les pages
while True:
    print(f"Scraping page : {driver.current_url}")
    scrape_page()

    try:
        # Cliquer sur le bouton "Page suivante"
        next_button = driver.find_element(By.CLASS_NAME, "next")
        ActionChains(driver).move_to_element(next_button).click().perform()
        time.sleep(3)  # Pause pour éviter les blocages
    except:
        print("Aucune page suivante trouvée. Fin du scraping.")
        break

# Sauvegarder les données dans un fichier CSV
df = pd.DataFrame({
    "Title": titles,
    "Company": companies,
    "Location": locations,
    "Date": dates,
    "Link": links,
    "Description": descriptions
})
df.to_csv("optioncarriere_jobs_full.csv", index=False)

print("Scraping terminé. Données sauvegardées dans 'optioncarriere_jobs_full1.csv'.")

# Fermer le driver
driver.quit()


Scraping page : https://www.optioncarriere.tn/emploi?s=&l=Tunisie
Scraping page : https://www.optioncarriere.tn/emploi?s=&l=Tunisie&p=2
Scraping page : https://www.optioncarriere.tn/emploi?s=&l=Tunisie&p=3
Scraping page : https://www.optioncarriere.tn/emploi?s=&l=Tunisie&p=4
Scraping page : https://www.optioncarriere.tn/emploi?s=&l=Tunisie&p=5
Scraping page : https://www.optioncarriere.tn/emploi?s=&l=Tunisie&p=6
Scraping page : https://www.optioncarriere.tn/emploi?s=&l=Tunisie&p=7
Scraping page : https://www.optioncarriere.tn/emploi?s=&l=Tunisie&p=8
Scraping page : https://www.optioncarriere.tn/emploi?s=&l=Tunisie&p=9
Scraping page : https://www.optioncarriere.tn/emploi?s=&l=Tunisie&p=10
Scraping page : https://www.optioncarriere.tn/emploi?s=&l=Tunisie&p=11
Scraping page : https://www.optioncarriere.tn/emploi?s=&l=Tunisie&p=12
Scraping page : https://www.optioncarriere.tn/emploi?s=&l=Tunisie&p=13
Scraping page : https://www.optioncarriere.tn/emploi?s=&l=Tunisie&p=14
Scraping page : htt

In [8]:
# Vérifier les longueurs des listes
print(f"Titles: {len(titles)}, Companies: {len(companies)}, Locations: {len(locations)}, Dates: {len(dates)}, Links: {len(links)}, Descriptions: {len(descriptions)}")

# Trouver la longueur maximale
max_length = max(len(titles), len(companies), len(locations), len(dates), len(links), len(descriptions))

# Compléter les listes pour qu'elles aient toutes la même longueur
titles += ["N/A"] * (max_length - len(titles))
companies += ["N/A"] * (max_length - len(companies))
locations += ["N/A"] * (max_length - len(locations))
dates += ["N/A"] * (max_length - len(dates))
links += ["N/A"] * (max_length - len(links))
descriptions += ["N/A"] * (max_length - len(descriptions))

# Créer le DataFrame après correction
df = pd.DataFrame({
    "Title": titles,
    "Company": companies,
    "Location": locations,
    "Date": dates,
    "Link": links,
    "Description": descriptions
})

# Sauvegarder dans un fichier CSV
df.to_csv("optioncarriere_jobs_full.csv", index=False)
print("Scraping terminé. Données sauvegardées dans 'optioncarriere_jobs_full.csv'.")


Titles: 2000, Companies: 2000, Locations: 2000, Dates: 2000, Links: 2000, Descriptions: 2000
Scraping terminé. Données sauvegardées dans 'optioncarriere_jobs_full.csv'.


# prétraitement

In [8]:
import numpy as np #numerical computing package
import pandas as pd #reading and manipulating dataframes 
import seaborn as sns #advanced visuals 
import matplotlib.pyplot as plt #basic visuals
import re #regular expressions
from googletrans import Translator
import matplotlib.pyplot as plt

In [9]:
data = pd.read_csv("optioncarriere_jobs_full.csv")  # Remplacez par votre fichier
print(data.head())

                                               Title          Company  \
0  Assistance Administrative, Financière et Compt...  Nextech Medical   
1  Gestionnaire locatif immobilier / Mise en loca...              NaN   
2                                 Réception d'Appels         Sunworld   
3  Sujet de PFE Élaboration et Mise en œuvre d'un...        NEXABLOOM   
4                              Analyste Devops (H/F)       Geothentic   

  Location             Date  \
0     Sfax  Il y a 20 jours   
1   Ariana  Il y a 20 jours   
2   Sousse  Il y a 20 jours   
3   Ariana  Il y a 20 jours   
4    Tunis  Il y a 20 jours   

                                                Link  \
0  https://www.optioncarriere.tn/jobad/tn63f4f57a...   
1  https://www.optioncarriere.tn/jobad/tn63f4f57a...   
2  https://www.optioncarriere.tn/jobad/tn8e79c807...   
3  https://www.optioncarriere.tn/jobad/tn34861c52...   
4  https://www.optioncarriere.tn/jobad/tne631751f...   

                                     

In [11]:
# describe the dataframe 
data.describe()

,Title,Company,Location,Date,Link,Description
count,2000,1703,2000,2000,2000,2000
unique,1158,1210,126,22,2000,1982
top,Manoeuvre manutentionnaire,Adecco,Tunis,Il y a 1 mois,https://www.optioncarriere.tn/jobad/tn63f4f57a...,Description de l'annonce:\nPour Accompagner so...
freq,58,13,377,560,1,7


In [12]:
data['Title_clean'] = data['Title'].str.lower().str.replace(r'[^a-zàâçéèêëîïôûùüÿñæœ\s]', '', regex=True)


In [13]:
data['Company'] = data['Company'].replace('N/A', 'Entreprise inconnue')

In [14]:
data['Location_clean'] = data['Location'].str.strip().str.lower()


In [15]:
from datetime import datetime, timedelta

# Fonction pour convertir une date relative en une date absolue
def convert_date(relative_date):
    if "jour" in relative_date:
        days = int(re.search(r'\d+', relative_date).group())
        return (datetime.today() - timedelta(days=days)).strftime('%Y-%m-%d')
    elif "mois" in relative_date:
        months = int(re.search(r'\d+', relative_date).group())
        return (datetime.today() - timedelta(days=months * 30)).strftime('%Y-%m-%d')
    return datetime.today().strftime('%Y-%m-%d')

data['Date_clean'] = data['Date'].apply(convert_date)

In [16]:
import re

# Nettoyage de la description
data['Description_clean'] = data['Description'].str.lower().apply(lambda x: re.sub(r'\W+', ' ', x))


In [17]:
competences = ['python', 'java', 'sql', 'machine learning', 'excel']

# Fonction pour extraire les compétences
def extract_competences(text):
    return [comp for comp in competences if comp in text]

data['Skills'] = data['Description_clean'].apply(extract_competences)


In [18]:
data.to_csv("donnees_pretraitees.csv", index=False)

In [3]:
data1 = pd.read_csv("donnees_pretraitees.csv") 

In [6]:
data1 = data1.drop(columns=["Link"])

In [7]:
# describe the dataframe 
data1.describe()

,Title,Company,Location,Date,Description,Title_clean,Location_clean,Date_clean,Description_clean,Skills
count,2000,1703,2000,2000,2000,2000,2000,2000,2000,2000
unique,1158,1210,126,22,1982,1139,126,21,1980,14
top,Manoeuvre manutentionnaire,Adecco,Tunis,Il y a 1 mois,Description de l'annonce:\nPour Accompagner so...,manoeuvre manutentionnaire,tunis,2024-12-08,description de l annonce pour accompagner son ...,[]
freq,58,13,377,560,7,58,377,600,8,1710


In [6]:
data1.drop_duplicates(inplace=True)

In [7]:
data1.Skills

0                    []
1             ['excel']
2                    []
3                    []
4       ['java', 'sql']
             ...       
1995                 []
1996                 []
1997                 []
1998                 []
1999                 []
Name: Skills, Length: 2000, dtype: object

In [8]:
data1

,Title,Company,Location,Date,Description,Title_clean,Location_clean,Date_clean,Description_clean,Skills
0,"Assistance Administrative, Financière et Compt...",Nextech Medical,Sfax,Il y a 20 jours,Vos principales missions :\nComptabilité (50%)...,assistance administrative financière et compta...,sfax,2024-12-18,vos principales missions comptabilité 50 saisi...,[]
1,Gestionnaire locatif immobilier / Mise en loca...,NaN,Ariana,Il y a 20 jours,Descriptif du poste :\nEn tant que Gestionnair...,gestionnaire locatif immobilier mise en location,ariana,2024-12-18,descriptif du poste en tant que gestionnaire l...,['excel']
2,Réception d'Appels,Sunworld,Sousse,Il y a 20 jours,Opportunité d’emploi – Rejoignez notre Centre ...,réception dappels,sousse,2024-12-18,opportunité d emploi rejoignez notre centre d ...,[]
3,Sujet de PFE Élaboration et Mise en œuvre d'un...,NEXABLOOM,Ariana,Il y a 20 jours,Objectifs du PFE :\nAnalyser le marché et la c...,sujet de pfe élaboration et mise en œuvre dune...,ariana,2024-12-18,objectifs du pfe analyser le marché et la cibl...,[]
4,Analyste Devops (H/F),Geothentic,Tunis,Il y a 20 jours,Geothentic Inc acteur majeur dans le domaine d...,analyste devops hf,tunis,2024-12-18,geothentic inc acteur majeur dans le domaine d...,"['java', 'sql']"
...,...,...,...,...,...,...,...,...,...,...
1995,Agent / Agente comptable,SOCIETE C T D TUNISIE S.A.R.L. MAINTE,Monastir,Il y a 1 mois,Référence : 5230/2024/2056 (Offre d'emploi)\nD...,agent agente comptable,monastir,2024-12-08,référence 5230 2024 2056 offre d emploi date d...,[]
1996,Assistant / Assistante photographe,STUDIO MONALIZA,Sousse,Il y a 1 mois,Référence : 5130/2024/2435 (Offre de stage)\nD...,assistant assistante photographe,sousse,2024-12-08,référence 5130 2024 2435 offre de stage date d...,[]
1997,Responsable d'atelier de piquage,STE S.T.G.S,"Moknine, Monastir",Il y a 1 mois,Référence : 5240/2024/1667 (Offre d'emploi)\nD...,responsable datelier de piquage,"moknine, monastir",2024-12-08,référence 5240 2024 1667 offre d emploi date d...,[]
1998,Ouvrier pâtissier / Ouvrière pâtissière,MAHA AOUICHI ART DES MATIeRES ALIMENTAIR,"Kalaâ Kebira, Sousse",Il y a 1 mois,Référence : 5130/2024/2433 (Offre d'emploi)\nD...,ouvrier pâtissier ouvrière pâtissière,"kalaâ kebira, sousse",2024-12-08,référence 5130 2024 2433 offre d emploi date d...,[]
